# 01 · Data: samples of each soil parameter

1. Reads the borehole file and keeps the **Downtown** boreholes (as V3).
2. For each target in `settings.toml → [cv] targets`, keeps the sample records with a value: record at most 3 m long, middle depth 0-15 m, value inside the target's `min`/`max`.
3. Marks the **11 external-holdout boreholes** of V3 (checked against V3's lock) and the development boreholes (those in V3's blockCV folds). Boreholes in neither have no V3 fold and are left out (their number is reported).
4. Saves one file per target in `data/processed/` (not in git) and summary tables without borehole codes in `outputs/01_data/`.

No gap filling: these are point values. SPT refusals and "non-plastic" entries were blanked in the CSV export, so they are missing, not zero (see the source review, §5).

**Run time:** under 1 minute.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])

## 1. Samples per target

In [ ]:
records = H.load_downtown(S)
holdout = H.load_v3_holdout(S)
folds = H.load_folds(S, H.block_sizes(S)[0])
P, O = H.processed_dir(S), H.out_dir(S, "01_data")
rows, info = [], {}
for t in S["cv"]["targets"]:
    smp, inf = H.make_samples(records, t, S)
    smp["set"] = H.assign_sets(smp, holdout.index, folds)
    smp.to_csv(P / f"samples_{t}.csv", index=False)
    info[t] = inf
    for s, g in smp.groupby("set"):
        rows.append({"target": t, "set": s, "boreholes": g.BH_Code.nunique(), "samples": len(g),
                     "mean": g.y.mean(), "sd": g.y.std(), "min": g.y.min(), "median": g.y.median(), "max": g.y.max()})
counts = pd.DataFrame(rows)
counts.to_csv(O / "sample_counts.csv", index=False)
display(pd.DataFrame(info).T)
display(counts.round(2))

## 2. Profiles with depth (development set)
Mean and interquartile range per 1 m slice. A model that only knows depth already captures this trend; the spatial models must beat it (notebook 03).

In [ ]:
ts = S["cv"]["targets"]
fig, axes = plt.subplots(1, len(ts), figsize=(3.2 * len(ts), 4.5), sharey=True)
prof = []
for ax, t in zip(np.atleast_1d(axes), ts):
    d = pd.read_csv(P / f"samples_{t}.csv")
    d = d[d.set == "development"]
    g = d.groupby(np.floor(d.depth_m)).y
    p = pd.DataFrame({"n": g.size(), "mean": g.mean(), "q25": g.quantile(0.25), "q75": g.quantile(0.75)})
    prof.append(p.assign(target=t))
    ax.fill_betweenx(p.index + 0.5, p.q25, p.q75, alpha=0.3)
    ax.plot(p["mean"], p.index + 0.5, marker="o")
    ax.set_title(t); ax.set_xlabel(S["targets"][t]["unit"])
np.atleast_1d(axes)[0].invert_yaxis(); np.atleast_1d(axes)[0].set_ylabel("depth (m)")
plt.tight_layout(); plt.savefig(O / "profiles_by_depth.png", dpi=200); plt.show()
pd.concat(prof).rename_axis("depth_from_m").to_csv(O / "profiles_by_depth.csv")

## 3. Value by USCS group (development set)
How strongly each parameter follows the Clay/Sand classes of the classification part.

In [ ]:
grp = {**{c: "Clay" for c in ["CL", "CH"]}, **{c: "Sand" for c in ["SC", "SM", "SP"]}}
rows = []
for t in ts:
    d = pd.read_csv(P / f"samples_{t}.csv")
    d = d[d.set == "development"].assign(group=lambda x: x.USCS.map(grp).fillna("other / missing"))
    rows.append(d.groupby("group").y.describe().assign(target=t))
by_uscs = pd.concat(rows).reset_index()
by_uscs.to_csv(O / "values_by_uscs_group.csv", index=False)
display(by_uscs.round(2))

In [ ]:
summary = {"targets": ts, "filtering": info, "holdout_boreholes": int(len(holdout))}
(O / "data_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 02.")